# Clase 178 — Chi-cuadrado: independencia y bondad de ajuste

El test χ² de Pearson en sus dos formas: **independencia** (dos categóricas cruzadas) y **bondad de ajuste** (observado vs distribución teórica). Verificamos el supuesto de frecuencias esperadas (Cochran), calculamos **Cramér's V** como effect size y usamos **Fisher exact** para 2×2 chicas.

Requiere: `numpy`, `pandas`, `scipy`, `matplotlib`.

## 1. Tabla de contingencia + test de independencia

`H₀`: las variables son independientes. Simulamos supervivencia dependiente de la clase (estilo Titanic).

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

n = 1200
clase = rng.choice(["1a", "2a", "3a"], size=n, p=[0.25, 0.20, 0.55])
p_surv = {"1a": 0.62, "2a": 0.43, "3a": 0.24}
survived = np.array([rng.random() < p_surv[c] for c in clase]).astype(int)

tabla = pd.crosstab(pd.Series(survived, name="survived"), pd.Series(clase, name="clase"))
print(tabla)
chi2, p, dof, expected = stats.chi2_contingency(tabla)
print(f"\nχ²={chi2:.2f}  dof={dof}  p={p:.2e}")
assert p < 0.001

## 2. Effect size: Cramér's V

El p-value dice si hay asociación; Cramér's V dice cuán fuerte. `V = √(χ² / (n·min(r-1, c-1)))`, entre 0 y 1.

In [ ]:
def cramers_v(chi2, table):
    nn = table.values.sum()
    r, c = table.shape
    return np.sqrt(chi2 / (nn * (min(r, c) - 1)))

V = cramers_v(chi2, tabla)
print(f"Cramér's V (clase) = {V:.3f}  (0.1 small, 0.3 medium, 0.5 large)")
assert 0 < V < 1

## 3. Supuesto de Cochran + G-test

El p-value asintótico es válido si al menos el 80 % de las celdas tiene `E ≥ 5`. El G-test (log-likelihood) se comporta mejor con celdas chicas.

In [ ]:
low = (expected < 5).sum()
print(f"celdas con E<5: {low} de {expected.size}")
g, p_g, dof_g, _ = stats.chi2_contingency(tabla, lambda_="log-likelihood")   # G-test
print(f"G-test: G={g:.2f}  dof={dof_g}  p={p_g:.2e}")

## 4. Fisher exact en tablas 2×2

Cuando alguna `E < 5` en una 2×2, Fisher da el p-value exacto. Comparamos la asociación con `sex` frente a la de `clase`.

In [ ]:
sexo = rng.choice(["M", "F"], size=n, p=[0.65, 0.35])
p_surv_sex = {"M": 0.19, "F": 0.73}
surv2 = np.array([rng.random() < p_surv_sex[s] for s in sexo]).astype(int)
t2 = pd.crosstab(pd.Series(surv2, name="survived"), pd.Series(sexo, name="sex"))
print(t2)

odds, p_fisher = stats.fisher_exact(t2)
chi2b, p_chi, _, _ = stats.chi2_contingency(t2)
Vb = cramers_v(chi2b, t2)
print(f"Fisher exact: OR={odds:.3f}  p={p_fisher:.2e}")
print(f"chi²:          p={p_chi:.2e}")
print(f"Cramér's V (sex) = {Vb:.3f}  vs (clase) = {V:.3f}")
assert Vb > V, "sexo tiene asociación más fuerte que clase" 

## 5. Bondad de ajuste: ¿el dado es justo?

Ahora el `H₀` es que la muestra sale de una distribución teórica (uniforme). `chisquare` exige que `f_exp.sum() == obs.sum()`.

In [ ]:
tiros = rng.choice([1, 2, 3, 4, 5, 6], size=600, p=[0.18, 0.16, 0.17, 0.17, 0.16, 0.16])
obs = np.array([np.sum(tiros == k) for k in range(1, 7)])
esp = np.full(6, tiros.size / 6)
gof = stats.chisquare(obs, f_exp=esp)
print("observado:", obs)
print(f"bondad de ajuste (dado justo): χ²={gof.statistic:.2f}  p={gof.pvalue:.3f}")
print("No rechazo: el sesgo es leve" if gof.pvalue > 0.05 else "Rechazo: dado cargado")

fig, ax = plt.subplots(figsize=(7, 4))
xk = np.arange(1, 7)
ax.bar(xk - 0.2, obs, width=0.4, label="observado")
ax.bar(xk + 0.2, esp, width=0.4, label="esperado (justo)")
ax.set_xlabel("cara del dado"); ax.legend(); ax.set_title("Bondad de ajuste del dado")
plt.tight_layout(); plt.show()

## Ejercicios

1. Aplicá la corrección de Yates (`correction=True`, default en 2×2) a la tabla `survived × sex` y comprobá cuánto cambia el p-value.
2. Contá qué porcentaje de celdas de `expected` viola `E ≥ 5`; si supera el 20 %, decidí entre Fisher o G-test.
3. Simulá un dado realmente cargado (`p=[0.30, 0.14, 0.14, 0.14, 0.14, 0.14]`) y verificá que ahora `chisquare` rechaza `H₀`.

## Conclusiones

- χ² de independencia y de bondad de ajuste usan la misma fórmula `Σ(O-E)²/E`; cambia de dónde salen las `E`.
- Verificá el supuesto de Cochran (`E ≥ 5`); si se viola, usá Fisher (2×2) o G-test.
- Cramér's V separa **tamaño de efecto** de **evidencia**: con `n` gigante un `p` minúsculo puede tener `V` chico.
- Asociación no es causalidad: cualquier confounder puede generar dependencia estadística.